## N-glycosylation from a GLYCAM06 named PDB file (retrieved from [GlyGen](https://www.glygen.org/glycan-search/))

Three glycans retrieved from GlyGen, each ending in a hydroxyl residue (`ROH`)
on the anomeric carbon. `fragment_from_pdb` reads each file with its
residues intact. `attach` bonds the anomeric carbon to ND2 of asparagine 60
of ubiquitin. Naming `O1` as the fragment's leaving atom removes the whole
hydroxyl, so `ROH` disappears and the GLYCAM residue names survive into the
written file, which is what a force field like GLYCAM06 keys on.

In [1]:
import logging, warnings
from rdkit import RDLogger

warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
RDLogger.DisableLog("rdApp.*")

In [2]:
from collections import Counter
from pathlib import Path

from mbuild.biopolymers import Protein, draw_fragment, fragment_from_pdb

mbuild_glycan = fragment_from_pdb("../glycans/glycam_G57321FI.pdb")
print([(residue.name, residue.resnum) for residue in mbuild_glycan.children], mbuild_glycan.n_particles, "atoms")
print("bond orders:", dict(Counter(d["bond_order"] for *_, d in mbuild_glycan.bonds(return_bond_order=True))))

[('ROH', 1), ('0VA', 2)] 30 atoms
bond orders: {1.0: 29, 2.0: 1}


`attach` names the glycan's atoms, and a PDB file's names are the builder's, not
ours. `draw_fragment` shows them: every atom labelled, one tint per residue with
a legend, and the names we pass in red. The anomeric carbon is `C1` of the sugar
residue (residue 2), and the hydroxyl that leaves is `O1` of the `ROH` residue with
its hydrogen `HO1`. Those three names are what the `attach` call below uses.

In [ ]:
draw_fragment(mbuild_glycan, highlight=["C1", "O1"], size=(900, 560))

Asparagine's amide nitrogen is neutral, so unlike a lysine it needs no
`deprotonate` first. `attach` removes one ND2 hydrogen itself.

In [3]:
mbuild_protein = Protein("../1ubq_protonated.pdb")
mbuild_protein.attach(
    mbuild_glycan,
    fragment_atom_name="C1",
    fragment_resnum=2,
    resnum=60,
    atom_name="ND2",
    leaving_atom_names="HD22",
    fragment_leaving_atom_names="O1",
)
bond_record, = mbuild_protein.bond_records()
bond_record

{'residue_names': ('ASN', '0VA'),
 'residue_numbers': (60, 77),
 'chain_ids': ('A', 'A'),
 'icodes': ('', ''),
 'atom_names': ('ND2', 'C1'),
 'leaving_atoms': (['HD22'], ['HO1', 'O1']),
 'bond_order': 1}

In [4]:
print([(r.name, r.resnum, r.hetatm) for r in mbuild_protein.residues()][75:])
print(mbuild_protein.n_particles, "atoms: 1231 protein + 30 glycan - HD22 - O1 - HO1 =", 1231 + 30 - 3)

[('GLY', 76, False), ('0VA', 77, True)]
1258 atoms: 1231 protein + 30 glycan - HD22 - O1 - HO1 = 1258


The same call for all three glycans, checking the written file each time.

In [9]:
display(mbuild_glycan)

<GLY 78 particles, 80 bonds, non-periodic, id: 134640078980448>

In [5]:
out = Path("../assets_cache")
for name in ("glycam_G57321FI", "glycam_G42666HT", "glycam_G15407YE"):
    mbuild_glycan = fragment_from_pdb(f"../glycans/{name}.pdb")
    mbuild_protein = Protein("../1ubq_protonated.pdb")
    mbuild_protein.attach(mbuild_glycan, fragment_atom_name="C1", fragment_resnum=2, resnum=60, atom_name="ND2",
                   leaving_atom_names="HD22", fragment_leaving_atom_names="O1")
    written = out / f"1ubq_{name}.pdb"
    mbuild_protein.save_pdb(written, overwrite=True)
    lines = written.read_text().splitlines()
    hetero = sorted({(line[17:20], int(line[22:26])) for line in lines if line.startswith("HETATM")}, key=lambda t: t[1])
    in_file = [(r.name, r.resnum) for r in mbuild_glycan.children if r.name != "ROH"]
    print(f"{name}: {[r.name for r in mbuild_glycan.children]} -> file has {hetero}, ROH present: {'ROH' in written.read_text()}")
    assert [name for name, _ in hetero] == [name for name, _ in in_file]

glycam_G57321FI: ['ROH', '0VA'] -> file has [('0VA', 77)], ROH present: False
glycam_G42666HT: ['ROH', '4YB', '0YB'] -> file has [('4YB', 77), ('0YB', 78)], ROH present: False
glycam_G15407YE: ['ROH', '4YB', '4YB', '0MB'] -> file has [('4YB', 77), ('4YB', 78), ('0MB', 79)], ROH present: False


Every atom of the product carries a bond order and a formal charge, so it exports.

In [ ]:
from openff.toolkit import Molecule

openff_molecule = Molecule.from_rdkit(mbuild_protein.to_rdkit(), allow_undefined_stereo=True)
print(openff_molecule.n_atoms, "atoms, net charge", openff_molecule.total_charge)